<div class="alert alert-block alert-success">

# Task 2 in Assessment 1
### Group 12: 

#### - Akhmad Fauzi Nurulhamzah
#### - [teammate]

Date: 16/10/2025

Environment: Python 3.13.7

</div>

<div class="alert alert-block alert-danger">
    
## Table of Contents

</div>

[1. Introduction](#Intro) <br>
[2. Importing Libraries](#libs) <br>
[3. Examining Input File](#examine) <br>
$\;\;\;\;$[3.1. Examining JSON File](#examine_json) <br>
$\;\;\;\;$[3.2. Examining Stopword File](#examine_stopwords) <br>
[4. Loading and Parsing File](#load) <br>
$\;\;\;\;$[4.1. Load and Read JSON File](#load_json) <br>
$\;\;\;\;$[4.2. Filter Review Data with at Least 50 Reviews](#filter_review) <br>
$\;\;\;\;$[4.3. Load and Read Stopwords File](#load_stopwords) <br>
[5. Generate The Unigram and Bigram](#gen_unigram) <br>
$\;\;\;\;$[5.1. Tokenization](#tokenize) <br>
$\;\;\;\;$[5.2. Token Grouping and Token Count](#token_count) <br>
$\;\;\;\;$[5.3. Remove Rare and Common Tokens](#remove_common_token) <br>
$\;\;\;\;$[5.4. Top 200 PMI bigrams](#pmi_bigrams) <br>
[6. Writing Output Files](#write) <br>
$\;\;\;\;$[6.1. Vocabulary List](#write-vocab) <br>
$\;\;\;\;$[6.2. Numerical Representations of All Tokens](#write-sparseMat) <br>
[7. Summary](#summary) <br>
[8. References](#Ref) <br>

<div class="alert alert-block alert-warning">
    
## 1.  Introduction  <a class="anchor" name="Intro"></a>

</div>

This assessment focuses on textual data processing, with the main objective of extracting, cleaning, and transforming raw text into a structured and analyzable format. The dataset is provided as a json file. This task represents the next stage in text data analysis, converting the extracted text into numerical representations that can be used for downstream modelling and machine learning tasks. These numerical forms are the standard input format for various Natural Language Processing (NLP) systems, including recommender systems, information retrieval algorithms, and machine translation models.

---

<div class="alert alert-block alert-warning">
    
## 2. Importing Libraries  <a class="anchor" name="libs"></a>

</div>

In this assessment, any python packages is permitted to be used. The following packages were used to accomplish the related tasks:

* **re:** Used to extract patterns and clean text data
* **math:** Used for rounding, thresholds
* **pandas:** Handling DataFrames for review data, merging, filtering
* **Counter:** Counts token frequencies efficiently
* **defaultdict:** Simplifies creation of dicts with default types
* **Path:** Simplifies file path handling and checking file
* **RegexpTokenizer:** Tokenization text
* **PorterStemmer:** Stemming algorithm to reduce words to their root form
* **BigramAssocMeasures:** Provides statistical scoring (like PMI) to evaluate bigram association strength
* **BigramCollocationFinder:** Finds frequently cooccurring word pairs (bigrams) in tokenized text

In [1]:
import re
import math
import pandas as pd

from collections import Counter, defaultdict
from pathlib import Path
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder

-------------------------------------

<div class="alert alert-block alert-warning">
    
## 3. Examining Input File <a class="anchor" name="examine"></a>

</div>

To simplify configuration, the code block below is used to define the input and output file paths. This approach ensures flexibility when running the notebook. for example, if a file is missing, renamed, or if the examiner executes the notebook from a different directory. In this case, all the required files are assumed to be located in the same folder as the main program. This task is implemented using Jupyter Notebook as the main development environment 

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

json_filename = "task1_012.json"
stopwords_filename = 'stopwords_en.txt'

output_vocab_filename  = "012_vocab.txt"
output_countvec_filename  = "012_countvec.txt"

---

<div class="alert alert-block alert-info">
    
### 3.1. Examining JSON File <a class="anchor" name="examine_json"></a>

</div>

The file task1_012.json is the exported output from Task 1. It is stored in JSON format and contains user review data that was previously extracted and transformed from multiple text and Excel files.

The JSON file is organized as a nested dictionary structure, where:
* Each key represents a unique product ID.
* Each product ID contains an object with a reviews key.
* The reviews value is a list of dictionaries, where each dictionary represents a single user review.

**The structure can be summarized as follows:**

{"product_id": {"reviews": [{"category": "<string>","reviewer_id": "<string>","rating": "<string>","review_title": "<string>","review_text": "<string>","attached_images": "<string>","product_id": "<string>","review_timestamp": "<string>","is_verified_purchase": "<string>","helpful_votes": "<string>"}]}}

---

<div class="alert alert-block alert-info">
    
### 3.2. Examining Stopword File <a class="anchor" name="examine_stopwords"></a>

</div>

According to the assignment description, a file named **stopwords_en.txt** was supposed to be provided. However, when attempting to open the provided link, access was restricted because it pointed to a Google Drive resource requiring special permission. Although we submitted an access request, the file remained inaccessible. Therefore, the **stopwords_en.txt** file is available in Moodle, Week 3 Exploratory Data Analysis (EDA) & Pre-Processing, and Data Quality & Anomalies materials, specifically within the “Foundations of Text Pre-Processing – 2.zip” file.

The file **stopwords_en.txt** contains a list of English stopwords, common words that are typically filtered out during text preprocessing because they carry minimal semantic meaning (e.g., “a”, “the”, “about”, “across”). These words are often removed to improve the accuracy and efficiency of text analysis, natural language processing (NLP), and machine learning models.

File Structure:
* The file is a plain text file (.txt) encoded in UTF-8.
* Each line contains a single stopword entry.
* There are no headers or additional metadata, only stopword terms.
* The stopwords are written in lowercase, with some containing contractions or punctuation marks (e.g., 'll, 'tis, 've).

---

<div class="alert alert-block alert-warning">
    
## 4. Loading and Parsing File <a class="anchor" name="load"></a>

</div>

In this section, we will load and parse data from the **task1_012.json** and **stopwords_en.txt** files. The parsed data will then be converted into a DataFrame for further processing and tokenization.

---

<div class="alert alert-block alert-info">
    
### 4.1. Load and Read JSON File <a class="anchor" name="load_json"></a>

</div>

In this section, we will open the JSON file and load its contents into the df_review DataFrame for further processing.

In [3]:
# Create an empty variable to store the JSON data
rows = []

# Reading data from the file
if Path(json_filename).exists():
    with open(json_filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    for parent, payload in data.items():
        for r in payload.get("reviews", []):
            txt = (r.get("review_text") or "").strip()
            rows.append({"parent_product_id": parent, "review_text": txt})

# Define the Review DataFrame
df_review =  pd.DataFrame(rows)

# Show Top 5 Data
df_review.head(5).style.hide(axis='index').set_properties(**{'text-align': 'left'})

parent_product_id,review_text
0440224764,good book.
0486475468,fun to do
0486475468,coloring is not just for kids any more. this is a wonderful way to unplug and have a little fun. like we used to.
0486475468,"these are lovely designs even before any colors are filled in. i very much enjoy the relaxation of coloring as i have zero talent for producing original paintings or drawings. i like doing these with good quality coloring pencils that have a rich pigment. i think that this book might be a thoughtful gift for any busy adult who wants something non-electronic to fill down time while waiting at doctor's appointments or for children to come out of an activity or even during business travel. i find it hard to relax enough to fall asleep sometimes after having a particularly stressful day at work, and this is something i can do with a little background music that doesn't contribute to insomnia like the light from a tv or computer screen."
0892877073,covers everything i needed


---

<div class="alert alert-block alert-info">
    
### 4.2. Filter Review Data with at Least 50 Reviews <a class="anchor" name="filter_review"></a>

</div>

In this section, we are required to extract the vocab and countvec lists only for reviews belonging to parent products that have at least 50 valid text reviews, excluding those labeled as 'none'.

In [4]:
# Filter data that have review_text = none
df_review = df_review.loc[df_review["review_text"].str.lower() != "none"].copy()

# Count many review for each parent_product_id 
df_review_count = df_review.groupby("parent_product_id")["review_text"].size()

# Filter parent_product_id that have review >= 50 
eligible_parents = set(df_review_count[df_review_count >= 50].index)

# Filter data that have review_text count < 50
df_review = df_review[df_review["parent_product_id"].isin(eligible_parents)].copy()

# Count valid parent_product_id
print(f"Eligible parents (>= 50 reviews): {len(eligible_parents)}")

# Show top 5 Data
df_review.head(5).style.hide(axis='index').set_properties(**{'text-align': 'left'})

Eligible parents (>= 50 reviews): 168


parent_product_id,review_text
b0002jn2eu,"i mean, it doesn't even taste salty."
b0002jn2eu,use in mower and car.
b0002jn2eu,"sea foam, non-oxy gas with a fuel stabilizer and your small engine will love you longtime. ""what do i get for 10?"" anything you want!"
b0002jn2eu,"i have used sea foam for many years with excellent results in motorcycles and engines that have been sitting a lot so that they do not run well. one big benefit to sea foam over many other fuel system cleaners is that it is also good for use in diesel engines, so if you have one of those around along with regular gas engines, this is good for both. and it works to improve and stabilize garbage alcohol adulterated fuels too. very important if you have machines that do a lot of sitting or keep gas around in a fuel can."
b0002jn2eu,"i mean, it doesn't even taste salty."


---

<div class="alert alert-block alert-info">
    
### 4.3. Load and Read Stopwords File <a class="anchor" name="load_stopwords"></a>

</div>

Before proceeding with further processing of the review data, we need to load and read the stopword file, which will be used in the next steps.

In [5]:
# Create an empty set to store the stopwords
df_stopword  = set()

# Reading data from the file
if Path(stopwords_filename).exists():
    with open(stopwords_filename, "r", encoding="utf-8") as f:
        for line in f:
            w = line.strip()
            if w and not w.startswith("#"):
                df_stopword .add(w.lower())

# Show 100 Data
print(f"Stopword: {list(df_stopword)[:100]}")

Stopword: ['four', "shouldn't", 'sorry', 'novel', 'over', 'qv', 'something', 'after', 'which', 'better', 'nowhere', 'less', 'q', 'during', 'exactly', 'overall', "we'll", 'quite', 'unto', 'tends', 'let', 'w', 'instead', 'might', 'beyond', 'ought', 'wonder', 'usually', "what's", 'gotten', 'awfully', "you're", 'sent', 'she', 'follows', 'go', "didn't", 'especially', 'with', 'should', 'certainly', 'okay', 'unless', 'what', 'went', 'thereby', "we've", 'because', 'never', 'even', 'everybody', 'elsewhere', "i'll", 'ie', 'yours', 'by', 'her', 'willing', "we'd", 'whom', 'help', 'under', 'do', 'across', 'seemed', 'they', 'appear', 'n', 'unfortunately', 'amongst', 'twice', 'done', "you've", 'sub', 'upon', 'thus', 'downwards', 'containing', 'oh', 'specify', 'besides', 'take', 'hello', 'three', "we're", 'self', 'am', 'how', 'taken', 'inc', 'furthermore', 'below', 'thence', 'hence', 'doing', 'hers', 'available', "doesn't", 'each', 'least']


---

<div class="alert alert-block alert-warning">
    
## 5. Generate The Unigram and Bigram  <a class="anchor" name="gen_unigram"></a>

</div>

---

<div class="alert alert-block alert-info">
    
### 5.1. Tokenization <a class="anchor" name="tokenize"></a>

</div>

Tokenization is a fundamental step in text processing used to produce unigrams (individual words). In this section, we apply a series of preprocessing steps: first, tokenizing the text using the regular expression "[a-zA-Z]+"; then removing all stopwords; followed by stemming using the Porter Stemmer algorithm. Finally, tokens with a length of fewer than three characters are removed to ensure the quality and relevance of the remaining vocabulary.

In [6]:
# The word tokenization process uses the following regular expression: "[a-zA-Z]+"
# This pattern extracts all continuous sequences of alphabetic characters (both uppercase and lowercase) and 
# returns them as a list of tokens. Non-letter characters such as numbers, punctuation, and symbols are excluded from the results.
df_review["tokens"] = df_review["review_text"].apply(lambda text: RegexpTokenizer(r"[a-zA-Z]+").tokenize((text or "").lower()))

# Now filter the tokens to remove all stopwords.
df_review["tokens"] = df_review["tokens"].apply(lambda toks: [t for t in toks if t not in df_stopword])

# Tokens should be stemmed using the Porter stemmer.
df_review["tokens"] = df_review["tokens"].apply(lambda toks: [PorterStemmer().stem(t) for t in toks])

# Tokens with a length less than 3 should be removed from the vocab.
df_review["tokens"] = df_review["tokens"].apply(lambda toks: [t for t in toks if len(t) >= 3])

df_review.head(5)

,parent_product_id,review_text,tokens
1158,b0002jn2eu,"i mean, it doesn't even taste salty.","[doesn, tast, salti]"
1159,b0002jn2eu,use in mower and car.,"[mower, car]"
1160,b0002jn2eu,"sea foam, non-oxy gas with a fuel stabilizer a...","[sea, foam, oxi, fuel, stabil, small, engin, l..."
1161,b0002jn2eu,i have used sea foam for many years with excel...,"[sea, foam, year, excel, result, motorcycl, en..."
1162,b0002jn2eu,"i mean, it doesn't even taste salty.","[doesn, tast, salti]"


-------------------------------------

<div class="alert alert-block alert-info">
    
### 5.2. Token Grouping and Token Count <a class="anchor" name="token_count"></a>

</div>

In this section, tokens are grouped and counted to identify their frequency of occurrence across all reviews. This process helps summarize the distribution of words in the dataset and provides insights into the most common terms used by reviewers.

In [7]:
# Build a dictionary mapping each parent_product_id to a set of unique tokens 
# appearing in all its reviews (used later for document frequency analysis)
parent_tokens_set = defaultdict(set)

for pid, grp in df_review.groupby("parent_product_id"):
    # union of tokens across all reviews of this parent
    union = set()
    for toks in grp["tokens"]:
        union.update(toks)
        
    parent_tokens_set[pid] = union

# Show the dictionary mapping
print("Dictionary mapping parent_product_id with unique tokens")
for i, (pid, tokens) in enumerate(parent_tokens_set.items()):
    print(f"{i+1}. {pid} - {list(tokens)[:10]}")  # show first 10 tokens only
    if i == 4:  # stop after 5 parents
        break
print('---')

# Join all tokens and calculate the frequency of each unique token
df_counts = Counter(t for toks in parent_tokens_set.values() for t in toks)

print('All tokens and frequency [:10]')
print(list(df_counts.items())[:10])

Dictionary mapping parent_product_id with unique tokens
1. b0002jn2eu - ['ethanol', 'compar', 'competit', 'privat', 'fals', 'gum', 'put', 'loosen', 'throat', 'wait']
2. b00068xcqu - ['compar', 'atv', 'safe', 'drag', 'drop', 'higher', 'proper', 'put', 'harley', 'cover']
3. b0006ix87s - ['flexabl', 'tini', 'competit', 'safe', 'sheen', 'put', 'camco', 'super', 'box', 'park']
4. b00080qhmm - ['compar', 'allway', 'higher', 'compat', 'put', 'box', 'dim', 'wait', 'suspect', 'june']
5. b0009iqzfm - ['hatchback', 'compar', 'weav', 'drag', 'whatsoev', 'washabl', 'super', 'wait', 'underperform', 'meguiar']
---
All tokens and frequency [:10]
[('ethanol', 4), ('compar', 69), ('competit', 9), ('privat', 2), ('fals', 11), ('gum', 6), ('put', 154), ('loosen', 8), ('throat', 4), ('wait', 59)]


---

<div class="alert alert-block alert-info">

### 5.3. Remove Rare and Common Tokens <a class="anchor" name="remove_common_token"></a>

</div>

In this step, we remove both rare tokens (words that appear very infrequently) and common tokens (words that appear too frequently) from the dataset. This filtering helps improve the quality of the vocabulary by retaining only the most informative words, which are more useful for subsequent text analysis tasks.

In [8]:
# Count the size of the parent products
num_parents = len(eligible_parents)
print(f"parent_product_id size = {num_parents}")
print('---')

# Set the threshold to words that appear in more than 95% of the parent products
common_cut = math.ceil(0.95 * num_parents)
print(f"95% x parent_product_id size = {common_cut}")

# Get tokens that appear more than 95%
context_dep_sw = {t for t, c in df_counts.items() if c > common_cut}
print(f"Context-dependent stopwords (> 95% parents): {len(context_dep_sw)}")
print(f'Common Tokens List: {list(context_dep_sw)[:10]}')
print('---')

# Rare tokens must be removed from the vocab with the threshold set to be words that appear in less than 5% of the parent products
rare_cut = max(1, math.floor(0.05 * num_parents)) 
print(f"5% x parent_product_id size = {rare_cut}")

# Get tokens that appear less than 5%
rare_terms = {t for t, c in df_counts.items() if c < rare_cut}
print(f"Rare terms (< 5% parents): {len(rare_terms)}")
print(f'Rare Tokens List [:10]: {list(rare_terms)[:10]}')
print('---')

# Apply common and rare filtering to tokens
df_review["tokens"] = df_review["tokens"].apply(lambda toks: [t for t in toks if t not in context_dep_sw and t not in rare_terms])

# Show top 5 data
print('Review Data After Filter')
df_review.head(5)

parent_product_id size = 168
---
95% x parent_product_id size = 160
Context-dependent stopwords (> 95% parents): 10
Common Tokens List: ['great', 'buy', 'product', 'recommend', 'time', 'easi', 'make', 'bought', 'work', 'good']
---
5% x parent_product_id size = 8
Rare terms (< 5% parents): 6736
Rare Tokens List [:10]: ['pontiac', 'densiti', 'refun', 'judg', 'maf', 'viehicl', 'nippl', 'consolegear', 'llegara', 'profil']
---
Review Data After Filter


,parent_product_id,review_text,tokens
1158,b0002jn2eu,"i mean, it doesn't even taste salty.","[doesn, tast]"
1159,b0002jn2eu,use in mower and car.,"[mower, car]"
1160,b0002jn2eu,"sea foam, non-oxy gas with a fuel stabilizer a...","[foam, fuel, stabil, small, engin, love]"
1161,b0002jn2eu,i have used sea foam for many years with excel...,"[foam, year, excel, result, motorcycl, engin, ..."
1162,b0002jn2eu,"i mean, it doesn't even taste salty.","[doesn, tast]"


---

<div class="alert alert-block alert-info">
    
### 5.4. Top 200 PMI bigrams <a class="anchor" name="pmi_bigrams"></a>

</div>

The first 200 meaningful bigrams must be included in the vocabulary using the PMI (Pointwise Mutual Information) measure. Additionally, the collocations should only be considered valid if both words appear within the same review.

In [9]:
# Top-200 PMI bigrams (compute on the eligible, filtered, stemmed tokens)
tokenized_reviews = df_review["tokens"].tolist()
print(f'Show Tokenized Review [:5]: \n{tokenized_reviews[:5]} \n---')

# Flatten all tokens from all reviews into a single list
all_tokens = [t for toks in tokenized_reviews for t in toks]
print(f'Show All Tokens [:10]: \n{all_tokens[:10]} \n---')

# Find bigrams using NLTK
finder = BigramCollocationFinder.from_words(all_tokens)
# ignore rare bigrams (appear < 3 times)
finder.apply_freq_filter(3)  

# Compute top 200 bigrams ranked by PMI
measures = BigramAssocMeasures()

# Score bigrams using PMI and get top 200
scored = finder.score_ngrams(measures.pmi)
top_bigrams = [f"{w1} {w2}" for (w1, w2), _ in sorted(scored, key=lambda x: x[1], reverse=True)[:200]]
print(f'Bigrams [:10]: \n{top_bigrams[:10]} \n---')

all_unigrams = set()
for toks in tokenized_reviews:
    all_unigrams.update(toks)

print(f'Unigrams [:10]: \n{list(all_unigrams)[:10]} \n---')

# Represent bigrams as space-separated tokens (per spec)
vocab_terms = sorted(all_unigrams.union(top_bigrams))
print(f'Vocab Terms [:10]: \n{list(vocab_terms)[:10]} \n---')

# Indexing: 0-based (change to +1 if your grader expects 1-based)
vocab_index = {tok: i for i, tok in enumerate(vocab_terms)}
print(f'Vocab Index [:10]: \n{list(vocab_index)[:10]} \n---')

Show Tokenized Review [:5]: 
[['doesn', 'tast'], ['mower', 'car'], ['foam', 'fuel', 'stabil', 'small', 'engin', 'love'], ['foam', 'year', 'excel', 'result', 'motorcycl', 'engin', 'sit', 'lot', 'run', 'big', 'benefit', 'foam', 'fuel', 'system', 'cleaner', 'diesel', 'engin', 'regular', 'engin', 'improv', 'stabil', 'garbag', 'fuel', 'import', 'machin', 'lot', 'sit', 'fuel'], ['doesn', 'tast']] 
---
Show All Tokens [:10]: 
['doesn', 'tast', 'mower', 'car', 'foam', 'fuel', 'stabil', 'small', 'engin', 'love'] 
---
Bigrams [:10]: 
['golf cart', 'buyer bewar', 'hyundai elantra', 'grand cheroke', 'stainless steel', 'potenti buyer', 'user friendli', 'muy bueno', 'elast band', 'licens plate'] 
---
Unigrams [:10]: 
['compar', 'proper', 'nope', 'collaps', 'dim', 'suspect', 'slid', 'flat', 'stabl', 'textur'] 
---
Vocab Terms [:10]: 
['aaa', 'abil', 'absolut', 'absorb', 'accept', 'access', 'accessori', 'accid', 'accident', 'accommod'] 
---
Vocab Index [:10]: 
['aaa', 'abil', 'absolut', 'absorb', 'acc

---

<div class="alert alert-block alert-warning">
    
## 6. Writing Output Files <a class="anchor" name="write"></a>

</div>

In this section, we will generate two output files:

* **012_vocab.txt**: containing all unique stemmed tokens, sorted in alphabetical order.
* **012_countvec.txt**: containing the numerical representations of all tokens, organized by parent_product_id and token index.

These files will serve as the final processed outputs for further analysis or model training.

---

<div class="alert alert-block alert-warning">
    
### 6.1. Vocabulary List <a class="anchor" name="write-vocab"></a>

</div>

The list of vocabulary terms should be written to a text file, sorted alphabetically, with their corresponding reference codes (indices) placed in front of each term. This vocabulary file serves as a reference for the sparse matrix stored in the next output file. After generating the vocab_terms, we can now export them to a .txt file for further use.

In [10]:
# Output the vocabulary file
# Each line contains a token followed by its reference index (e.g., "word:1"),
# sorted alphabetically. This file serves as a reference for the count vector file.
with open(output_vocab_filename, "w", encoding="utf-8") as f:
    for tok in vocab_terms:
        f.write(f"{tok}:{vocab_index[tok]}\n")

# Display success message after exporting the file
print(f"Wrote vocab: {output_vocab_filename} (size= {len(vocab_terms)})")

Wrote vocab: 012_vocab.txt (size= 1973)


---

<div class="alert alert-block alert-warning">
    
### 6.2. Numerical Representations of All Tokens <a class="anchor" name="write-sparseMat"></a>

</div>

For writing sparse matrix for a paper, we firstly calculate the frequency of words for that paper ....

In [11]:
# Variable to collect each parent count-vector row
parent_rows = []

# Loop through each parent product
for pid, grp in df_review.groupby("parent_product_id"):

    # store token frequency counts for this parent
    cnt = Counter()

    # Loop through all reviews belonging to this parent
    for toks in grp["tokens"]:
        
        # Count unigrams (single words)
        for t in toks:
            if t in vocab_index:
                cnt[vocab_index[t]] += 1
                
        # Count bigrams (pairs of consecutive words)
        for bg in [f"{toks[i]} {toks[i+1]}" for i in range(len(toks) - 1)]:
            if bg in vocab_index:
                cnt[vocab_index[bg]] += 1
   
    # build sparse row: "parent_product_id,idx:freq,idx:freq, ..."
    parts = [pid]
    for idx in sorted(cnt):
        parts.append(f"{idx}:{cnt[idx]}")
    parent_rows.append(",".join(parts))

# Write all parent rows into output file
with open(output_countvec_filename, "w", encoding="utf-8") as f:
    for line in parent_rows:
        f.write(line + "\n")

# Display success message after exporting the file
print(f"Wrote countvec: {output_countvec_filename} (parents={len(parent_rows)})")

Wrote countvec: 012_countvec.txt (parents=168)


-------------------------------------

<div class="alert alert-block alert-warning">
    
## 7. Summary <a class="anchor" name="summary"></a>

</div>

In this task, we successfully exported two key output files: **012_vocab.txt** and **012_countvec.txt**, which will be used for subsequent text vectorization and model training processes. We began by cleaning and preparing the review text data through several steps, then we tokenized, stemmed, and filtered the tokens by removing both context-independent stopwords (from stopwords_en.txt) and context-dependent stopwords (based on word frequency across parent products).

Next, we generated the top 200 PMI-ranked bigrams and combined them with the unigrams to create a complete vocabulary list, which was saved as 012_vocab.txt. Using this vocabulary, we computed term frequency counts (for both unigrams and bigrams) across each parent product, resulting in the sparse representation file 012_countvec.txt.

Overall, the preprocessing and feature extraction pipeline ensures that the text data is clean, normalized, and ready for downstream NLP or machine learning tasks, such as clustering, topic modeling, or sentiment classification.

-------------------------------------

<div class="alert alert-block alert-warning">
    
## 8. References <a class="anchor" name="Ref"></a>

</div>

[1] Collocations and PMI for bigrams: Sample usage for collocations. https://www.nltk.org/howto/collocations.html,  Accessed 22/10/2025.

[2] English Stop Words. https://github.com/stopwords-iso/stopwords-en/blob/master/stopwords-en.txt,  Accessed 22/10/2025.

